# Como eu me enganei sem perceber?

**Nível 3 — Avançado** · Trilha Wizz Lab

> Este nível inteiro é sobre o pesquisador, não sobre a estratégia.


Ele mede o efeito de ter testado muitas coisas, de ter olhado o mesmo histórico muitas
vezes e de ter parado de procurar quando o resultado ficou bonito.


---

In [ ]:
%matplotlib inline

# No Colab, instala o pacote direto do GitHub. Localmente, não faz nada.
import importlib.util, subprocess, sys

if importlib.util.find_spec("wizzlab") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "git+https://github.com/Gustavofthiesen/wizz-lab.git"], check=True)

from wizzlab import brand, data, metrics, scorecard
from wizzlab.theme import aplicar_tema
from wizzlab import charts

aplicar_tema()
print("Wizz Lab pronto · paleta:", brand.SERIE_PRINCIPAL, brand.SERIE_COMPARACAO)

In [ ]:
import numpy as np, pandas as pd
est = data.gerar_estrategia(semente=42)
r = est.trades.r_multiple.values

## 1. Quantas observações você tem de verdade?

Quase toda fórmula de erro-padrão supõe independência. Se os resultados forem
autocorrelacionados, os intervalos ficam estreitos demais — e você se convence sem motivo.

In [ ]:
print(f"N bruto ................. {len(r)}")
print(f"N efetivo ............... {metrics.evidencia.effective_sample_size(r):.0f}")
print(f"t-stat ingênuo .......... {metrics.evidencia.t_stat(r):.2f}")
print(f"t-stat Newey-West (HAC) . {metrics.evidencia.newey_west_t(r):.2f}")
q, p = metrics.evidencia.ljung_box(r)
print(f"Ljung-Box Q = {q:.2f}, p = {p:.3f}")

Nestes dados simulados a autocorrelação é fraca, então os dois t batem. **Em dados reais
raramente batem** — e quando o t despenca ao corrigir, o t ingênuo estava contando a mesma
informação várias vezes.

O bootstrap também precisa respeitar a dependência:

In [ ]:
for nome, fn in [("iid", metrics.evidencia.iid_bootstrap),
                 ("moving block", metrics.evidencia.moving_block_bootstrap),
                 ("stationary", metrics.evidencia.stationary_bootstrap)]:
    v = fn(r, n_boot=3000)
    print(f"{nome:14} → IC 95% [{np.quantile(v, .025):+.3f} ; {np.quantile(v, .975):+.3f}]")

## 2. Sharpe, corrigido de verdade

O Sharpe bruto ignora assimetria, curtose e — sobretudo — quantas vezes você tentou.

In [ ]:
sr_diario = metrics.risco.sharpe(est.diario) / np.sqrt(252)
n = len(est.diario)
skew = metrics.trade.skewness(est.diario)
kurt = metrics.trade.excess_kurtosis(est.diario) + 3

psr = metrics.evidencia.probabilistic_sharpe(sr_diario, n, skew, kurt)
mintrl = metrics.evidencia.min_track_record_length(sr_diario, skew, kurt)

print(f"Sharpe anualizado ......... {metrics.risco.sharpe(est.diario):.2f}")
print(f"PSR (P[Sharpe real > 0]) .. {psr:.1%}")
print(f"MinTRL .................... {mintrl:,.0f} pregões  (~{mintrl/252:.1f} anos)")

**MinTRL é o antídoto mais direto contra track record curto.** Ele responde: quanto
histórico seria preciso para sustentar este Sharpe com 95% de confiança?

Agora o Deflated Sharpe — que desconta o fato de você ter testado várias configurações:

In [ ]:
for n_trials in (1, 10, 50, 200):
    dsr = metrics.evidencia.deflated_sharpe(sr_diario, n, max(n_trials, 2),
                                            var_sharpes=0.0004, skew=skew, kurt=kurt)
    print(f"{n_trials:4d} tentativas → DSR = {dsr:.1%}")

O mesmo Sharpe. A mesma estratégia. **O que muda é quantas vezes você procurou** — e isso
muda a conclusão.

## 3. PBO: a métrica mais desconfortável do manual

Ela não avalia a estratégia. Avalia o **processo de seleção**.

Vamos gerar 40 configurações que são **puro ruído** — nenhuma tem edge — e perguntar o que
acontece se escolhermos a melhor pelo backtest.

In [ ]:
rng = np.random.default_rng(0)
ruido_puro = rng.normal(0, 1, size=(600, 40))   # 600 períodos, 40 configurações

melhor = int(np.argmax(ruido_puro.mean(axis=0)))
print(f"Melhor configuração no IS: #{melhor}, "
      f"retorno médio {ruido_puro[:, melhor].mean():+.4f}")
print(f"PBO (CSCV): {metrics.generalizacao.pbo_cscv(ruido_puro):.2f}")

Testando 40 coisas sem valor nenhum, alguma parece boa. O PBO próximo de 0,5 diz
exatamente isso: **escolher pelo backtest não é melhor que sortear.**

E o Reality Check confirma pelo outro lado:

In [ ]:
print(f"White's Reality Check, p = {metrics.generalizacao.reality_check(ruido_puro):.3f}")
print(f"Hansen SPA, p ............. {metrics.generalizacao.hansen_spa(ruido_puro):.3f}")

p alto = a melhor regra encontrada é compatível com o que se acharia testando N regras sem
valor nenhum.

> **O corolário prático:** registre quantas configurações você testou. Sem esse número,
> nenhuma dessas correções funciona — e é por isso que o manual trata *Research Trials
> Count* como métrica, não como burocracia.

## 4. O indicador por dentro

Até aqui avaliamos a estratégia pronta. Agora o sinal que a alimenta — são coisas
diferentes, e separá-las evita consertar a peça errada.

In [ ]:
sinal_df = data.gerar_sinal(n=3000, ic_verdadeiro=0.045, semente=11)

print(f"IC (Pearson) .......... {metrics.sinal.information_coefficient(sinal_df.score, sinal_df.retorno_futuro):.4f}")
print(f"Rank IC (Spearman) .... {metrics.sinal.rank_ic(sinal_df.score, sinal_df.retorno_futuro):.4f}")
print(f"Monotonicidade ........ {metrics.sinal.signal_monotonicity(sinal_df.score, sinal_df.retorno_futuro):+.2f}")

In [ ]:
fig, ax = charts.quantis_sinal(sinal_df.score, sinal_df.retorno_futuro,
                               fonte="Sinal simulado com IC = 0,045 · wizz-lab")
fig

**Calibragem que evita vergonha:** em ações, IC de 0,02 a 0,05 já é um sinal de valor.
Quem mostra IC de 0,40 deve procurar o vazamento de informação antes de procurar a
explicação econômica.

E note quanta amostra foi preciso (3000 observações) para que um IC de 0,045 produzisse
uma progressão de quantis visível.

In [ ]:
decay = pd.Series({1: 0.048, 3: 0.041, 5: 0.030, 10: 0.018, 21: 0.006, 42: -0.002})
fig, ax = charts.signal_decay(decay, fonte="Sinal simulado · wizz-lab")
fig

O horizonte de maior retorno condicional indica a **meia-vida do sinal**. Um pico isolado
em um horizonte estranho merece suspeita, não comemoração.

## 5. A pergunta que quase ninguém faz

O indicador novo acrescenta algo **além** dos filtros que já existem? A maioria dos
indicadores "novos" é uma combinação linear dos antigos.

In [ ]:
rng = np.random.default_rng(3)
antigo_a = sinal_df.score.values
antigo_b = rng.normal(0, 1, len(sinal_df))
# Um "novo" indicador que é 85% o antigo A disfarçado:
novo = 0.85 * antigo_a + 0.15 * rng.normal(0, 1, len(sinal_df))

metrics.sinal.incremental_signal_value(
    novo, [antigo_a, antigo_b], sinal_df.retorno_futuro.values).round(4)

IC bruto respeitável, IC incremental perto de zero. O indicador "novo" não trouxe
informação nova — trouxe a informação antiga com outro nome.

## 6. Estabilidade: o edge está morrendo?

In [ ]:
fig, ax = charts.rolling(est.diario, janela=252,
                         titulo="O edge persiste ou é episódico?",
                         subtitulo="Retorno médio diário em janela de 252 pregões",
                         fonte="Dados simulados · wizz-lab")
fig

In [ ]:
print(metrics.sinal.edge_decay(est.diario).round(6).to_string())
print()
print(metrics.sinal.rolling_stability(est.diario).round(5).to_string())
print()
print("Dependência do melhor regime:",
      f"{metrics.sinal.regime_dependency_score(est.trades.r_multiple, est.trades.regime):.1%}")

Se o P&L depende demais de um regime, você não tem uma estratégia — tem uma aposta em um
regime. O que não é necessariamente errado, desde que esteja declarado.

---

## O que levar deste nível

1. N bruto não é N efetivo. Corrija por dependência antes de acreditar no t.
2. MinTRL põe um número em "esse track record é curto demais".
3. O DSR desconta quantas vezes você tentou — então **conte as tentativas**.
4. PBO acima de 0,5: escolher pelo backtest é pior que sortear.
5. IC de 0,02–0,05 já é sinal. IC de 0,40 é vazamento.
6. Pergunte sempre o valor **incremental** de um indicador novo.

**No notebook 4:** tudo isso junto, em ordem, como scorecard.

---

### Bloco de transparência

**Natureza:** educacional · **Dados:** simulados e reprodutíveis por semente ·
**Código:** aberto em [wizz-lab](https://github.com/Gustavofthiesen/wizz-lab)

Este material apresenta um processo de estudo, com finalidade educacional. Não
constitui recomendação individualizada, oferta ou promessa de retorno. Premissas podem
estar erradas e resultados passados não garantem resultados futuros.